In [1]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import Subset
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import torch.nn as nn
import numpy as np
from sklearn.model_selection import train_test_split
import torchvision
from torchvision import transforms, datasets
from torchvision.models import resnet18, ResNet18_Weights
%matplotlib inline

In [2]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),   
    transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])]) 

dataset = datasets.ImageFolder(root="C://Users//USER//Downloads//archive (2)//256_ObjectCategories", transform=transform)

remove_class_name = '256_ObjectCategories'
remove_idx = dataset.class_to_idx[remove_class_name]

In [3]:
valid_indices = [i for i, label in enumerate(dataset.targets) if label != remove_idx]
dataset.samples = [dataset.samples[i] for i in valid_indices]
dataset.targets = [dataset.targets[i] for i in valid_indices]
dataset.imgs = dataset.samples

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [5]:
target = dataset.targets
indices = np.arange(len(dataset))

train_idx, temp_idx = train_test_split(indices,test_size=0.3,stratify=target,random_state=42)
temp_targets = [target[i] for i in temp_idx]
val_idx, test_idx = train_test_split(temp_idx,test_size=0.5,stratify=temp_targets,random_state=42)

train_dataset = Subset(dataset, train_idx)
valid_dataset = Subset(dataset, val_idx)
test_dataset  = Subset(dataset, test_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [6]:
print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(valid_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Train samples: 21424
Validation samples: 4591
Test samples: 4592


In [7]:
weights = ResNet18_Weights.DEFAULT


In [8]:
model = resnet18(weights=weights)
in_features = model.fc.in_features
model.fc = nn.Sequential(nn.Dropout(p=0.3), nn.Linear(in_features, 258))

model = model.to(device)

In [9]:
criterion = nn.CrossEntropyLoss()

learning_rate = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate,weight_decay =0.005, momentum=0.9)
model.to(device)

def validation_test(model, valid_loader):
    total_loss = 0.0
    total_samples = 0
    
    with torch.no_grad():
        model.eval()
        for image, label in valid_loader:
            image, label = image.to(device), label.to(device)
            pred_label = model(image)
            loss = criterion(pred_label, label)
            
            # Multiply by current batch size to get the total raw loss for this batch
            batch_size = image.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size
            
    # Divide total loss by total number of images
    return total_loss / total_samples

In [10]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    for batch_idx,(image,label) in enumerate(train_loader):
       
        image = image.to(device)
        label = label.to(device)

        pred_label = model(image)
        loss = criterion(pred_label,label)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if batch_idx%500==0:
            validation_loss = validation_test(model,val_loader)   
            model.train()
            print(f"EPOCHS[{epoch+1}/{num_epochs}] : TRAINING LOSS {loss.item():.4f} : VALIDATION LOSS {validation_loss:.4f}")  

EPOCHS[1/10] : TRAINING LOSS 5.8644 : VALIDATION LOSS 5.7861
EPOCHS[1/10] : TRAINING LOSS 2.4329 : VALIDATION LOSS 1.9657
EPOCHS[2/10] : TRAINING LOSS 1.6533 : VALIDATION LOSS 2.2535
EPOCHS[2/10] : TRAINING LOSS 2.0745 : VALIDATION LOSS 2.3025
EPOCHS[3/10] : TRAINING LOSS 1.7214 : VALIDATION LOSS 2.2757
EPOCHS[3/10] : TRAINING LOSS 1.8130 : VALIDATION LOSS 2.4161
EPOCHS[4/10] : TRAINING LOSS 1.8106 : VALIDATION LOSS 2.2701
EPOCHS[4/10] : TRAINING LOSS 2.1779 : VALIDATION LOSS 2.6665
EPOCHS[5/10] : TRAINING LOSS 1.7380 : VALIDATION LOSS 2.5116
EPOCHS[5/10] : TRAINING LOSS 1.5787 : VALIDATION LOSS 2.7106
EPOCHS[6/10] : TRAINING LOSS 1.9553 : VALIDATION LOSS 2.4720
EPOCHS[6/10] : TRAINING LOSS 2.8038 : VALIDATION LOSS 2.5371
EPOCHS[7/10] : TRAINING LOSS 1.8569 : VALIDATION LOSS 2.5519
EPOCHS[7/10] : TRAINING LOSS 2.3464 : VALIDATION LOSS 2.5551
EPOCHS[8/10] : TRAINING LOSS 1.9049 : VALIDATION LOSS 2.6954
EPOCHS[8/10] : TRAINING LOSS 2.0627 : VALIDATION LOSS 3.1276
EPOCHS[9/10] : TRAINING 

In [11]:
def test_model(model,dataset,device):
    model.eval()
    correct = 0
    total = 0
    total_loss = 0
    with torch.no_grad():
        for batch_idx,(test_image, test_label) in enumerate(dataset):

            image = test_image.to(device)
            label = test_label.to(device)

            pred_label = model(image)
            
            loss = criterion(pred_label,label)
            total_loss += loss.item()
            _, predicted = torch.max(pred_label, dim=1)
            
            total += label.size(0)
            correct += (predicted == label).sum().item()

    return correct/total, total_loss/len(dataset)      

In [12]:
training_accuracy, training_loss = test_model(model,train_loader,device)
validation_accuracy, validation_loss = test_model(model,val_loader,device)
testing_accuracy, testing_loss = test_model(model,test_loader,device)

print(f"Training Accuracy: {training_accuracy*100:.2f}%")
print(f"Validation Accuracy: {validation_accuracy*100:.2f}%")
print(f"Testing Accuracy: {testing_accuracy*100:.2f}%")

Training Accuracy: 62.75%
Validation Accuracy: 48.51%
Testing Accuracy: 48.41%


In [13]:
print(f"Training Loss: {training_loss:.4f}")
print(f"Validation Loss: {validation_loss:.4f}")
print(f"Testing Loss: {testing_loss:.4f}")

Training Loss: 1.7555
Validation Loss: 2.4088
Testing Loss: 2.4119
